# Etapa 1 — Exploración del dataset

Carga del PhiUSIIL Phishing URL Dataset y chequeos básicos. Solo vamos a usar la columna `URL`.

In [ ]:
import pandas as pd
import tldextract

RUTA_CSV = "PhiUSIIL_Phishing_URL_Dataset.csv"

# El CSV viene con BOM UTF-8; con "utf-8-sig" la primera columna queda como "FILENAME"
df = pd.read_csv(RUTA_CSV, encoding="utf-8-sig")

# Chequeo: label debería tener solo los valores 0 y 1
valores_label = set(df["label"].unique())
if valores_label != {0, 1}:
    print(f"ATENCIÓN: 'label' tiene valores inesperados: {valores_label}")

# En el dataset original label = 1 es LEGÍTIMA y label = 0 es PHISHING.
# Invertimos para trabajar siempre con phishing = 1.
df["phishing"] = 1 - df["label"]

In [ ]:
# Forma del dataset
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")

In [ ]:
# Columnas disponibles (solo usaremos URL; el resto son features precalculadas que ignoramos)
print(list(df.columns))

In [ ]:
# Balance de clases
nombres_clase = {0: "legítima", 1: "phishing"}
balance = pd.DataFrame({
    "cantidad": df["phishing"].value_counts(),
    "proporción": df["phishing"].value_counts(normalize=True).round(4),
}).rename(index=nombres_clase)
balance

In [ ]:
# 5 URLs de ejemplo de cada clase
for clase, nombre in nombres_clase.items():
    print(f"--- {nombre} ---")
    ejemplos = df.loc[df["phishing"] == clase, "URL"].sample(5, random_state=42)
    for url in ejemplos:
        print(url)
    print()

In [ ]:
# Dominios registrables únicos (misma definición que usa features.py)
from features import dominio_registrable

dominios = df["URL"].map(dominio_registrable)
print(f"Dominios registrables únicos (total): {dominios.nunique():,}")
for clase, nombre in nombres_clase.items():
    print(f"Dominios registrables únicos ({nombre}): {dominios[df['phishing'] == clase].nunique():,}")

# Etapa 2 — Prueba de extract_features

Probamos la función de `features.py` con algunas URLs a mano (todavía no con el dataset completo).

In [ ]:
from features import extract_features

urls_prueba = [
    "https://www.bna.com.ar",
    "http://bna-homebanking-verificar.xyz/login",
    "https://www.afip.gob.ar/",                            # sufijo .gob.ar
    "http://192.168.0.1:8080/paypal/login.php?id=1&t=2",   # IP, puerto, marca y parámetros
    "https://bit.ly/3xYz12",                               # acortador
    "https://storage.googleapis.com/algo",                 # dominio oficial de Google
    "http://paypal-login-seguro.com",                      # marca fuera de su dominio
]

# Una columna por URL para compararlas lado a lado
pd.DataFrame([extract_features(u) for u in urls_prueba], index=urls_prueba).T

**Nota: posible artefacto del dataset.** En PhiUSIIL muchas URLs legítimas son solo la página de inicio (`https://www.dominio.com`) y el phishing suele tener path. Por eso `longitud_path`, `profundidad_path`, `usa_https` y `longitud_url` pueden separar las clases por cómo se armó el dataset y no por una señal real de phishing. Si más adelante las métricas dan casi perfectas, es la primera causa a revisar.

# Etapa 3 — Features del dataset completo

Aplicamos `extract_features` a todas las URLs y guardamos el resultado en `data/features.parquet`.

In [ ]:
import time

import numpy as np

from features import dominio_registrable, extract_features

inicio = time.perf_counter()
X = pd.DataFrame([extract_features(u) for u in df["URL"]])
y = df["phishing"].reset_index(drop=True)
dominio = df["URL"].map(dominio_registrable).reset_index(drop=True)
duracion = time.perf_counter() - inicio

print(f"Forma de X: {X.shape}")
print(f"Tiempo: {duracion:.1f} s ({len(X) / duracion:,.0f} URLs/s)")

In [ ]:
# Controles de calidad (solo se muestran; no se elimina nada)
nulos = X.isna().sum()
print(f"Valores nulos: {nulos.sum()}")
if nulos.sum():
    print(nulos[nulos > 0])

infinitos = pd.Series(np.isinf(X.to_numpy(dtype=float)).sum(axis=0), index=X.columns)
print(f"Valores infinitos: {infinitos.sum()}")
if infinitos.sum():
    print(infinitos[infinitos > 0])

print(f"Filas con dominio vacío: {(dominio == '').sum()}")

urls = df["URL"].reset_index(drop=True)
print(f"Filas con URL duplicada: {urls.duplicated().sum():,}")
print(f"URLs distintas que se repiten: {urls[urls.duplicated(keep=False)].nunique():,}")
etiquetas_por_url = df.groupby("URL")["phishing"].nunique()
print(f"URLs duplicadas con etiquetas distintas: {(etiquetas_por_url > 1).sum():,}")

**Columnas que NO son features** en `data/features.parquet`:
- `URL`: se guarda solo para inspeccionar los errores del modelo en la etapa 4.
- `phishing`: el target (1 = phishing).
- `dominio`: dominio registrable, se usa como grupo en `GroupShuffleSplit`.

Al entrenar, hay que excluir estas tres columnas de `X`.

In [ ]:
from pathlib import Path

RUTA_FEATURES = Path("data") / "features.parquet"
RUTA_FEATURES.parent.mkdir(exist_ok=True)

df_features = X.assign(URL=urls, phishing=y, dominio=dominio)
df_features.to_parquet(RUTA_FEATURES, index=False)

print(f"Guardado en {RUTA_FEATURES}")
print(f"Forma: {df_features.shape}")
print(f"Tamaño: {RUTA_FEATURES.stat().st_size / 1e6:.1f} MB")